In [10]:
#fetches the cards, obtaining the card data
import requests

resp = requests.get("https://db.ygoprodeck.com/api/v7/cardinfo.php")
cards = resp.json()["data"]  # list of ~13,000 card dicts

import json
with open("all_cards.json", "w") as f:
    json.dump(cards, f)

In [11]:
import json

with open("all_cards.json") as f:
    cards = json.load(f)

# Collect every unique key that appears across all cards
all_keys = set()
for card in cards:
    all_keys.update(card.keys())

print(f"Total unique keys found: {len(all_keys)}")
for key in sorted(all_keys):
    print(key)

Total unique keys found: 23
archetype
atk
attribute
banlist_info
card_images
card_prices
card_sets
def
desc
frameType
humanReadableCardType
id
level
linkmarkers
linkval
monster_desc
name
pend_desc
race
scale
type
typeline
ygoprodeck_url


In [12]:
# Use sentence-transformers locally — runs on your own machine, no per-call cost, no rate limit.
import json

# Load the card data you already fetched and saved from the YGOPRODeck API
with open("all_cards.json") as f:
    cards = json.load(f)

import os
# Sanity check — confirm working directory and that the file actually loaded from there
print(os.getcwd())
print(os.path.exists("all_cards.json"))

/Volumes/T7 Shield/ai eng/sematic search
True


In [18]:
#Embedding step
from sentence_transformers import SentenceTransformer

# Load a small, free, local embedding model (downloads once, then runs offline)
model = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast, free, good enough for this

# Turn each card dict into a single text string to embed
def card_to_text(card):
    parts = [f"{card['name']} ({card['type']})"]

    if "race" in card:
        parts.append(f"Race: {card['race']}")
    if "attribute" in card:
        parts.append(f"Attribute: {card['attribute']}")
    if "level" in card:
        parts.append(f"Level: {card['level']}")
    if "atk" in card:
        parts.append(f"ATK: {card['atk']}")
    if "def" in card:
        parts.append(f"DEF: {card['def']}")
    if "scale" in card:
        parts.append(f"Pendulum Scale: {card['scale']}")
    if "linkval" in card:
        parts.append(f"Link Rating: {card['linkval']}")
    if "archetype" in card:
        parts.append(f"Archetype: {card['archetype']}")

    parts.append(f"Effect: {card['desc']}")

    if "pend_desc" in card:
        parts.append(f"Pendulum Effect: {card['pend_desc']}")

    return ". ".join(parts)

# Build the list of texts, one per card
texts = [card_to_text(c) for c in cards]
# Embed all card texts at once (batched internally for speed)
embeddings = model.encode(texts, batch_size=64, show_progress_bar=True)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/227 [00:00<?, ?it/s]

In [20]:
def card_to_metadata(card):
    images = card.get("card_images", [])
    image_url = images[0].get("image_url", "") if images else ""

    atk = card.get("atk")
    if atk is None:
        atk = -1

    defe = card.get("def")
    if defe is None:
        defe = -1

    return {
        "name": card["name"],
        "type": card["type"],
        "archetype": card.get("archetype", ""),
        "atk": atk,
        "def": defe,
        "image_url": image_url
    }
    
metadatas = [card_to_metadata(c) for c in cards]

In [21]:
for i, meta in enumerate(metadatas):
    for k, v in meta.items():
        if v is None:
            print(f"Card index {i}: field '{k}' is None -> card name: {cards[i]['name']}")

In [22]:
#loading into chromadb, the vector database of choice
import chromadb

client = chromadb.PersistentClient(path="./yugioh_db")

# Delete the old collection since the text/metadata format changed
try:
    client.delete_collection("cards")
except:
    pass

collection = client.create_collection("cards")

ids = [str(c["id"]) for c in cards]

batch_size = 5000
for i in range(0, len(cards), batch_size):
    collection.add(
        ids=ids[i:i+batch_size],
        documents=texts[i:i+batch_size],
        embeddings=embeddings[i:i+batch_size].tolist(),
        metadatas=metadatas[i:i+batch_size]
    )
    print(f"Added batch {i} to {i+batch_size}")

Added batch 0 to 5000
Added batch 5000 to 10000
Added batch 10000 to 15000


In [23]:
query_emb = model.encode(["Spiritual Beast Tamer Winda"]).tolist()
results = collection.query(query_embeddings=query_emb, n_results=1)

print(results["documents"][0][0])   # the embedded text
print(results["metadatas"][0][0])   # the metadata dict

Spiritual Beast Tamer Winda (Effect Monster). Race: Psychic. Attribute: WIND. Level: 4. ATK: 1600. DEF: 1800. Archetype: Ritual Beast. Effect: If this card in its owner's possession is destroyed by an opponent's card (by battle or card effect): You can Special Summon 1 "Ritual Beast" monster from your Deck or Extra Deck, ignoring its Summoning conditions. You can only Special Summon "Spiritual Beast Tamer Winda(s)" once per turn.
{'archetype': 'Ritual Beast', 'name': 'Spiritual Beast Tamer Winda', 'atk': 1600, 'def': 1800, 'type': 'Effect Monster', 'image_url': 'https://images.ygoprodeck.com/images/cards/65193366.jpg'}


In [24]:
print(collection.count()) 

14516


In [7]:
query_emb = model.encode(["monster that special summons itself from the graveyard"])
results = collection.query(query_embeddings=query_emb.tolist(), n_results=5)

for name, dist in zip(results["metadatas"][0], results["distances"][0]):
    print(name["name"], dist)

The Creator 0.505770742893219
Royal Prison 0.5333552360534668
Heraldry Reborn 0.573615312576294
Jowgen the Spiritualist 0.6282885670661926
Card of Safe Return 0.6311872005462646


In [25]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "what is Spiritual Beast Tamer Winda's attack?"}]
})
print(response["messages"][-1].content)

Spiritual Beast Tamer Winda's attack is 1600.


In [8]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_ollama import ChatOllama
from sentence_transformers import SentenceTransformer
import chromadb
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

# Reconnect to your existing persistent Chroma DB and embedding model
client = chromadb.PersistentClient(path="./yugioh_db")
collection = client.get_or_create_collection("cards")
model = SentenceTransformer("all-MiniLM-L6-v2")

@tool
def search_yugioh_cards(query: str) -> str:
    """Search the Yu-Gi-Oh card database for cards relevant to the query."""
    query_emb = model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_emb, n_results=5)
    return "\n\n".join(results["documents"][0])

llm = ChatOllama(
    model="qwen2.5:7b-instruct",
    temperature=0.3,
)

agent = create_react_agent(llm, tools=[search_yugioh_cards])

response = agent.invoke({
    "messages": [{"role": "user", "content": "What monster special summons itself from the graveyard?"}]
})
print(response["messages"][-1].content)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

It seems that none of the cards listed in the search results actually special summon themselves from the graveyard. However, there are a few cards that allow you to special summon other monsters from your graveyard under certain conditions. Here are a few examples:

1. **Spirit of the Pharaoh**: This card can be Special Summoned from the graveyard, and when it does, you can Special Summon up to 4 Level 2 or lower Zombie-Type Normal Monsters from your graveyard.

2. **Infernity Mirage**: This card cannot be Special Summoned from the graveyard, but if you have no cards in your hand, you can Tribute this card to select 2 "Infernity" monsters in your graveyard and Special Summon them.

3. **Royal Prison**: This trap card prevents monsters from being Special Summoned from the graveyard.

If you're looking for a card that can special summon itself, you might want to check for specific cards like "Self-Rescue" or "Self-Revival," but it's important to note that such cards are not commonly foun